# Intercoder Reliability Analysis

This notebook evaluates the consistency of the manual coding procedure by comparing the researcher's classifications with independent classifications generated by ChatGPT (GPT-6). A random sample of approximately 20% of the dataset for each company is independently classified using the same predefined coding framework. Agreement is assessed separately for Leadership_context and Information_valence using Cohen's kappa (Cohen, 1960) and exact percentage agreement. This analysis provides a reliability assessment of the coding procedure and supports the methodological transparency of the study.

## 1. Dataset Validation

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

# Reproducibility
RANDOM_SEED = 42
SAMPLE_FRACTION = 0.20

# Input files
files = {
    "adidas": "../data/input/preparation/adidas.xlsx",
    "puma": "../data/input/preparation/puma.xlsx",
    "asics": "../data/input/preparation/asics.xlsx",
    "mizuno": "../data/input/preparation/mizuno.xlsx",
}

# Required columns
required_columns = [
    "ID",
    "Text",
    "Text_type",
    "Leadership_context_human",
    "Information_valence_human",
    "Communication_type",
    "Source_type",
    "Company",
    "Country",
    "Year",
    "Covid",
    "Communicator_role",
    "Communicator_name"
]

# Output directory
output_dir_report = Path("../data/input/preparation/quality_reports")
output_dir_report.mkdir(parents=True, exist_ok=True)

# Process each company
for company, file_path in files.items():

    df = pd.read_excel(file_path)

    output_file = output_dir_report / f"{company}_quality_report.txt"

    with open(output_file, "w", encoding="utf-8") as f:

        def log(message=""):
            print(message, file=f)

        log("=" * 60)
        log(f"DATA QUALITY REPORT: {company.upper()}")
        log("=" * 60)

        # 1. Dataset overview
        log("\n1. DATASET OVERVIEW")
        log(f"Dataset size: {df.shape}")
        log(f"Number of rows: {len(df)}")
        log(f"Number of columns: {len(df.columns)}")

        log("\nColumns:")
        for col in df.columns:
            log(f"- {col}")

        # 2. Check required columns
        log("\n2. REQUIRED COLUMNS CHECK")

        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]

        if missing_columns:
            log(f"Missing columns: {missing_columns}")
        else:
            log("All required columns are present.")

        # 3. Missing values
        log("\n3. MISSING VALUES")

        for col in required_columns:

            if col not in df.columns:
                continue

            missing = df[col].isna().sum()
            percentage = df[col].isna().mean() * 100

            log(
                f"{col}: {missing} missing values "
                f"({percentage:.2f}%)"
            )

        # 4. Unique values
        log("\n4. UNIQUE VALUES")

        for col in required_columns:

            if col not in df.columns:
                continue

            unique_count = df[col].nunique(dropna=True)

            log(f"{col}: {unique_count} unique values")

        # 5. Category distributions
        log("\n5. CATEGORY DISTRIBUTIONS")

        for col in required_columns:

            if col not in df.columns:
                continue

            # Skip free-text and identifier columns
            if col in ["ID", "Text"]:
                continue

            log(f"\n{col}:")
            log(df[col].value_counts(dropna=False).to_string())

        # 6. Duplicate checks
        log("\n6. DUPLICATE CHECKS")

        if "ID" in df.columns:
            duplicate_ids = df["ID"].dropna().duplicated().sum()
            log(f"Duplicate IDs: {duplicate_ids}")

        if "Text" in df.columns:
            duplicate_texts = df["Text"].dropna().duplicated().sum()
            log(f"Duplicate texts: {duplicate_texts}")

        log("\n" + "=" * 60)
        log("END OF REPORT")
        log("=" * 60)

    print(f"Report saved: {output_file}")

Report saved: ../data/input/preparation/quality_reports/adidas_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/puma_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/asics_quality_report.txt
Report saved: ../data/input/preparation/quality_reports/mizuno_quality_report.txt


## 2. Random Sampling

In [3]:
company_data = {}

for company, file_path in files.items():

    df = pd.read_excel(file_path)

    # Standardize whitespace in relevant string columns
    for col in [
        "ID",
        "Leadership_context_human",
        "Information_valence_human"
    ]:
        df[col] = df[col].map(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    # Select eligible records
    eligible = df.dropna(
        subset=[
            "ID",
            "Text",
            "Leadership_context_human",
            "Information_valence_human"
        ]
    ).copy()

    eligible = eligible[
        eligible["Text"].astype(str).str.strip().ne("")
    ].copy()

    company_data[company] = eligible

    print(
        f"{company.upper()}: "
        f"{len(df)} original records, "
        f"{len(eligible)} eligible records, "
        f"{len(df) - len(eligible)} excluded records."
    )

ADIDAS: 219 original records, 219 eligible records, 0 excluded records.
PUMA: 105 original records, 105 eligible records, 0 excluded records.
ASICS: 206 original records, 206 eligible records, 0 excluded records.
MIZUNO: 62 original records, 62 eligible records, 0 excluded records.


In [4]:
company_samples = {}
sampling_records = []

for company, df in company_data.items():

    sample = df.sample(
        frac=SAMPLE_FRACTION,
        random_state=RANDOM_SEED
    ).sort_values("ID").reset_index(drop=True)

    company_samples[company] = sample

    sampling_records.append({
        "Company": company,
        "Eligible_records": len(df),
        "Sample_records": len(sample),
        "Sample_percentage": round(
            len(sample) / len(df) * 100, 2
        ) if len(df) else np.nan
    })

sampling_summary = pd.DataFrame(sampling_records)
display(sampling_summary)


,Company,Eligible_records,Sample_records,Sample_percentage
0,adidas,219,44,20.09
1,puma,105,21,20.00
2,asics,206,41,19.90
3,mizuno,62,12,19.35


## 3. Preparation of the blinded coding sample

In [6]:
# Output directory
output_dir_ai = Path("../data/input/preparation/ai_results")
output_dir_ai.mkdir(parents=True, exist_ok=True)

ai_input_columns = [
    "ID",
    "Text",
    "Text_type",
    "Communication_type",
    "Source_type",
    "Company",
    "Country",
    "Year",
    "Covid",
    "Communicator_role",
    "Communicator_name"
]

company_references = {}

for company, sample in company_samples.items():

    safe_name = company.lower()

    # Keep only columns that exist in the dataset
    columns_to_export = [
        col for col in ai_input_columns
        if col in sample.columns
    ]

    ai_input = sample[columns_to_export].copy()

    # Researcher's original classifications
    reference = sample[
        [
            "ID",
            "Leadership_context_human",
            "Information_valence_human"
        ]
    ].copy()

    company_references[company] = reference

    ai_input.to_excel(
        output_dir_ai  / f"AI_{safe_name}.xlsx",
        index=False
    )

    reference.to_excel(
        output_dir_ai  / f"AI_reference_{safe_name}.xlsx",
        index=False
    )

    print(f"{company.upper()}: blinded input exported.")

ADIDAS: blinded input exported.
PUMA: blinded input exported.
ASICS: blinded input exported.
MIZUNO: blinded input exported.


## 4. Independent AI Coding

In [ ]:
ai_results_by_company = {}

for company in company_samples:

    ai_path = output_dir_ai / f"{company}_AI_results.xlsx"

    if not ai_path.exists():
        raise FileNotFoundError(
            f"AI results file not found: {ai_path}"
        )

    ai = pd.read_excel(ai_path)

    required_ai_columns = [
        "ID",
        "Leadership_context_AI",
        "Information_valence_AI"
    ]

    missing_columns = [
        col for col in required_ai_columns
        if col not in ai.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{company}: missing columns {missing_columns}"
        )

    if ai["ID"].isna().any() or ai["ID"].duplicated().any():
        raise ValueError(
            f"{company}: missing or duplicate IDs."
        )

    expected_ids = set(company_references[company]["ID"])
    actual_ids = set(ai["ID"])

    if expected_ids != actual_ids:
        raise ValueError(
            f"{company}: ID mismatch. "
            f"Missing: {expected_ids - actual_ids}; "
            f"Extra: {actual_ids - expected_ids}"
        )

    ai_results_by_company[company] = ai.copy()

    print(f"{company.upper()}: AI results validated.")

## 5. Data Matching and Comparison

In [ ]:
comparisons_by_company = {}

for company in company_samples:

    comparison = company_references[company].merge(
        ai_results_by_company[company][
            [
                "ID",
                "Leadership_context_AI",
                "Information_valence_AI"
            ]
        ],
        on="ID",
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    if not comparison["_merge"].eq("both").all():
        raise ValueError(
            f"{company}: unmatched records found."
        )

    comparison = comparison.drop(columns="_merge")
    comparison["Company"] = company

    comparisons_by_company[company] = comparison

    comparison.to_excel(
        output_dir / f"{company}_coding_comparison.xlsx",
        index=False
    )

    print(f"{company.upper()}: classifications matched.")

## 6. Cohen's Kappa and Exact Agreement

In [ ]:
def calculate_reliability(
    comparison,
    researcher_col,
    ai_col,
    allowed_categories
):
    researcher = comparison[researcher_col].astype("string").str.strip()
    ai = comparison[ai_col].astype("string").str.strip()

    valid = (
        researcher.notna()
        & ai.notna()
        & researcher.ne("")
        & ai.ne("")
    )

    researcher = researcher[valid].astype(str)
    ai = ai[valid].astype(str)

    if len(researcher) == 0:
        return {
            "N_valid_pairs": 0,
            "Exact_agreement_pct": np.nan,
            "Cohens_kappa": np.nan
        }

    observed = set(researcher) | set(ai)
    unexpected = observed - set(allowed_categories)

    if unexpected:
        raise ValueError(
            f"Unexpected coding labels: {sorted(unexpected)}"
        )

    agreement = (
        researcher.to_numpy() == ai.to_numpy()
    ).mean() * 100

    kappa_value = cohen_kappa_score(researcher, ai)

    return {
        "N_valid_pairs": len(researcher),
        "Exact_agreement_pct": round(agreement, 2),
        "Cohens_kappa": (
            round(float(kappa_value), 4)
            if np.isfinite(kappa_value)
            else np.nan
        )
    }

## 7. Disagreement Analysis

In [ ]:
for company, comparison in comparisons_by_company.items():

    for researcher_col, ai_col in [
        ("Leadership_context_human", "Leadership_context_AI"),
        ("Information_valence_human", "Information_valence_AI")
    ]:

        researcher = comparison[researcher_col].astype("string").str.strip()
        ai = comparison[ai_col].astype("string").str.strip()

        valid = (
            researcher.notna()
            & ai.notna()
            & researcher.ne("")
            & ai.ne("")
        )

        disagreements = comparison.loc[
            valid & (researcher != ai)
        ].copy()

        print(
            f"{company.upper()} — {researcher_col}: "
            f"{len(disagreements)} disagreements"
        )

        disagreements.to_excel(
            output_dir
            / f"{company}_{researcher_col}_disagreements.xlsx",
            index=False
        )

## 8. Export and Report Results

In [ ]:
# Combine all company comparisons
all_comparisons = pd.concat(
    list(comparisons_by_company.values()),
    ignore_index=True
)

all_comparisons.to_excel(
    output_dir / "all_companies_coding_comparison.xlsx",
    index=False
)

sampling_summary.to_csv(
    output_dir / "sampling_summary_by_company.csv",
    index=False
)

print("Reliability analysis completed.")
display(reliability_results)